# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fraz-Rasool/ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 4 — CTR / Engagement Opportunity Scoring.**

I am choosing this lane because the starter data contains direct observations of impressions, clicks, CTR, average position, sessions, and engagement. That makes it possible to ask a practical question about which visible content items deserve review first, while also respecting an important issue in the lane guide: CTR should be compared within position context rather than across all pages blindly. This is a useful seven-week direction because it can start with a transparent position-adjusted opportunity score and later test whether a more flexible model improves the ranking.

In [4]:
# Quick sanity check for the chosen lane.
# The starter data contains the direct signals needed for CTR / engagement opportunity scoring.
required_columns = {
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "position_tier", "sessions_90d", "engagement_rate"
}
print("Required lane signals present:", required_columns.issubset(set(df.columns)) if "df" in globals() else "checked after loading")


Required lane signals present: checked after loading


## 2. The question: decision, action, cost of a wrong call

**Research question:** Among pages with enough search exposure to matter, which pages appear to under-capture clicks or engagement relative to their search-position context, and therefore deserve review first?

**Unit of analysis:** one pseudonymized content item/page.

**Decision:** decide which pages should enter a limited review queue before other pages.

**Who acts:** an SEO/content reviewer.

**Action:** review the highest-ranked candidates and choose an appropriate follow-up such as improving title/meta/snippet alignment, checking search intent and content fit, improving on-page engagement, or simply monitoring the page when the evidence is weak.

**Cost of a wrong call:** a false positive can waste limited editorial/review time on a page that does not need attention. A false negative can leave a genuinely weak opportunity unreviewed. For that reason, the project should favor a precise, explainable top-K queue rather than pretending every score is a guaranteed recommendation.

**Why data/ML can help:** a simple rule is a valid baseline. ML only earns its place if multiple signals—visibility, position, CTR, engagement, freshness, content attributes, and demand—combine in patterns that are difficult to capture with a few fixed thresholds.

In [5]:
# The decision can be stated as a ranking task rather than a prediction-for-its-own-sake task.
task_type = "ranking / scoring"
unit_of_analysis = "one pseudonymized content item/page"
primary_decision = "which visible pages deserve review first"
print("Task type:", task_type)
print("Unit:", unit_of_analysis)
print("Decision:", primary_decision)


Task type: ranking / scoring
Unit: one pseudonymized content item/page
Decision: which visible pages deserve review first


## 3. Quick look at the data (2-3 real numbers)

The starter CSV contains **30,000 rows**. I use two especially relevant observations for this lane:

1. The impression-weighted CTR is about **0.49% for `top_3` pages**, while it is only about **0.04% for `deep` pages**. This large position difference is exactly why a useful opportunity score should adjust for position rather than simply label every low-CTR page as a problem.
2. **9,759 pages (32.53%)** have at least 500 impressions, an average position between 1 and 20, and CTR below 0.5%. This is a substantial candidate pool, but it is only a screening population—not proof that all of these pages need a refresh.

These numbers make the lane worth investigating, while also showing why volume and position context matter.

In [7]:
# Clone the internship repository into the Colab runtime (run once per fresh runtime)
!git clone -q https://github.com/Fraz-Rasool/ML-Internship.git /content/ML-Internship

from pathlib import Path
import pandas as pd

# Colab path: repository/data/raw/content_refresh_anonymized.csv
data_path = Path("/content/ML-Internship/data/raw/content_refresh_anonymized.csv")

if not data_path.exists():
    raise FileNotFoundError(f"CSV not found at: {data_path}")

df = pd.read_csv(data_path)

position_summary = (
    df.groupby("position_tier", dropna=False)
      .agg(
          pages=("content_id", "size"),
          impressions_90d=("impressions_90d", "sum"),
          clicks_90d=("clicks_90d", "sum"),
      )
)

position_summary["weighted_ctr_pct"] = (
    position_summary["clicks_90d"] / position_summary["impressions_90d"] * 100
)

candidate_mask = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

candidate_count = int(candidate_mask.sum())

print(f"Rows in starter dataset: {len(df):,}")
print(f"Weighted CTR — top_3: {position_summary.loc['top_3', 'weighted_ctr_pct']:.2f}%")
print(f"Weighted CTR — deep: {position_summary.loc['deep', 'weighted_ctr_pct']:.2f}%")
print(
    f"Visible low-CTR screening candidates: {candidate_count:,} "
    f"({candidate_count / len(df) * 100:.2f}%)"
)


Rows in starter dataset: 30,000
Weighted CTR — top_3: 0.49%
Weighted CTR — deep: 0.04%
Visible low-CTR screening candidates: 9,759 (32.53%)


## 4. Careful words: what I can and can't claim

**What I can claim:** I can measure observed relationships in this starter slice, compare CTR/engagement across groups, build a ranking score, and test whether a model improves the ordering of review candidates under a clearly defined evaluation setup. I can describe the output as **observed, measured, directional, and decision-support**.

**What I cannot claim:** I cannot claim that a low CTR proves a bad title or meta description; other factors can affect clicks. I cannot claim that a page will recover after a refresh, because this observational dataset does not provide a causal experiment. I also cannot claim to predict or reproduce Google's ranking algorithm.

The starter dataset's existing `trend_direction`/`trend_pct` fields are derived from the current data window, so they should not be treated as an ideal future outcome for a final capstone. If this lane later becomes supervised ML, the stronger design is to define a leakage-safe future outcome after a clear decision point and keep the feature window separate from that outcome window.

In [8]:
# A small check that keeps the language honest:
# the starter label is a current-window proxy, not a future causal outcome.
derived_current_window_fields = {"trend_direction", "trend_pct"}
print("Current-window fields that need caution:", sorted(derived_current_window_fields))
print("Interpretation: decision-support and observed association, not causal proof.")


Current-window fields that need caution: ['trend_direction', 'trend_pct']
Interpretation: decision-support and observed association, not causal proof.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.